在 PyTorch 中进行多 GPU 或多节点分布式训练时，NCCL（NVIDIA Collective Communications Library，NVIDIA 集体通信库）是官方极力推荐的后端，它是实现高效、可扩展训练的关键一环。

简单来说，NCCL 是一个专门为 NVIDIA GPU 优化的通信库，它提供了一系列像 all-reduce、broadcast 这样的“集体通信”功能，让不同 GPU 之间可以高效地同步数据和交换梯度。你可以把它想象成一个为 GPU 们设计的“高速对讲系统”，确保它们在大规模并行训练中步调一致。

例如，在使用预训练模型进行微调时，我们可以将输入数据分成多个批次，并将其分配给不同的GPU进行处理。然后，NCCL可以帮助我们将每个GPU上的输出合并起来，以便我们可以对其进行后处理并更新模型参数。

此外，NCCL还可以帮助我们在不同节点之间的GPU之间进行通信。这对于分布式训练非常有用，因为它允许我们在多个计算机上同时运行相同的模型，并使用所有可用的GPU进行训练。
[Understanding Multi GPU Communication and Nvidia NCCL for finetuning models](https://tensorfuse.io/docs/blogs/multi_gpu_communication_while_training)

## 为什么 PyTorch 训练首选 NCCL？

在 PyTorch 的分布式训练包 torch.distributed 中，主要有 NCCL、Gloo 和 MPI 三种后端。它们各有侧重，选择哪个主要取决于你的硬件环境

- NCCL
    多 GPU/多机训练的首选。能充分利用 NVLink、InfiniBand 等高速互联技术，提供超高带宽和低延迟的通信，是 DistributedDataParallel (DDP) 的默认后端
    
- Gloo
    适用于 CPU 训练，或在没有 NVIDIA GPU、NCCL 出现异常时作为备选方案。在 GPU 上的通信速度明显慢于 NCCL
    
- MPI
    一个更通用的通信标准，但在 PyTorch 中作为可选后端，需要从源码编译才能使用，且对 GPU 的支持有限
    


## 如何在 PyTorch 中使用 NCCL
- 只需在初始化分布式进程组时指定 backend=‘nccl’ 即可

In [ ]:
import  torch.distrituted as dist

dist.init_process_group(backend='nccl',
                       init_method='env://',
                        world_size=args.world_size,
                        rank=args.rank
                       )
'''
之后，当你使用 torch.nn.parallel.DistributedDataParallel (DDP) 包装模型时，
PyTorch 就会自动调用 NCCL 来完成梯度同步等关键通信操作。
'''

## 常见配置与故障排除

### 指定通信网卡：

如果你的机器有多个网络接口（比如以太网和 InfiniBand），需要手动指定 NCCL 使用哪一个，避免选错导致通信失败或性能低下

In [ ]:
export NCCL_SOCKET_IFNAME=eth0  # 将其替换为实际的网卡名称，如 ib0

### 开启调试信息：

当训练遇到挂起或报错时，设置这个变量可以打印出 NCCL 的详细初始化信息和警告，是定位问题的第一把钥匙

In [ ]:
export NCCL_DEBUG=INFO

### 应对超时错误：

在初始化大模型或网络不稳定时，可能会遇到“NCCL watchdog timeout”错误。可以适当增加 PyTorch 的初始化超时时间来避免。

In [ ]:
python -m torch.distributed.run --nproc_per_node=NUM_GPUS --timeout=3600 your_train_script.py

### 利用 NVLink 加速：

如果你的 GPU 之间通过 NVLink 桥接器连接，可以强制 NCCL 优先使用这条“高速公路”，能极大提升单机多卡通信效率

In [ ]:
export NCCL_P2P_LEVEL=NVL

## NCCL 不止在 PyTorch 中

NCCL 的强大之处在于它作为一个基础库，被广泛应用于各类 GPU 加速的通信场景。例如，在大语言模型训练框架 Megatron-LM 中，其核心的张量并行（Tensor Parallelism）和数据并行（Data Parallelism）策略，就高度依赖 NCCL 提供的 all-gather、reduce-scatter 等集合通信算子来高效合并计算结果

| 特性 | NVLink | InfiniBand (IB) |
| :--- | :--- | :--- |
| 主要作用范围 | 节点内 / 机柜内 (Intra-node / Intra-rack) | 节点间 / 集群级 (Inter-node / Cluster-wide) |
| 连接对象 | GPU ↔ GPU (或 GPU ↔ CPU) | 服务器 ↔ 服务器 (通过交换机) |
| 最大带宽 (2026标准) | ~900 GB/s - 1.8 TB/s (双向聚合，如 NVL72) | 400 Gb/s - 1.6 Tb/s (单链路，约 50-200 GB/s) |
| 延迟 | 极低 (亚微秒级，接近片上通信) | 低 (微秒级，受限于物理距离和交换机跳数) |
| 拓扑结构 | 全互联 (Full Mesh) 或 专用背板 (如 NVL72) | Fat-Tree (胖树), Dragonfly 等网络拓扑 |
| 所有者/生态 | NVIDIA 私有协议 (仅限 NVIDIA GPU) | 开放标准 (由 InfiniBand Trade Association 定义，NVIDIA/Mellanox 主导) |
| 典型应用场景 | 模型并行 (Tensor Parallelism)，大显存池化 | 数据并行 (Data Parallelism)，流水线并行跨节点通信 |
| 传输距离 | 极短 (几厘米到几米，通常在机柜内) | 长 (可达数百米甚至公里，通过光纤) |



# 完整的 NCCL 分布式训练示例

## 基础训练脚本：nccl_distributed_train.py

In [ ]:
import os
import argparse
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nultiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.util.data import DataLoade,Dataset
from torch.util.data.distributed import DistributedSampler
import numpy as np

# 1.创建一个简单的模型
class SimpleModel(nn.Moudle):
    def __init__(self,input_size=784,hidden_size=256,num_classes=10):
        super().__init__();
        self.fc1 = nn.Linear(input_size,hidden_size)
        self.relu = nn.Relu()
        self.fc2 = nn.Linear(hidden_size,hidden_size)
        self.fc3 = nn.Linear(hidden_size,num_classes)
        
    def forward(self,x):
        x = x.view(x.size(0),-1) # 展平输入
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
# 2.创建一个模拟数据集
class RandomDataset(Dataset):
    '''生成随机数据用于演示'''
    def __init__(self,num_samples=10000,input_size=784,num_classes=10):
        self.data = torch.randn(num_samples,1,28,28)
        self.labels = torch.randint(0,num_classes,(num_samples,))
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        return self.data[idx],self.lables[idx]
    
# 3.分布式训练的核心设置
def setup(rank,world_size,backend='nccl'):
    '''初始化分布式环境'''
    # 设置环境变量（也可以通过命令行传递参数）
    os.environ['MASTER_ADDR'] = os.environ.get('MASTER_ADDR','localhost')
    os.environ['MASTER_PORT'] = os.environ.get('MASTER_PORT','29500')
    
    # 初始化进程组
    # backend='nccl' 指定使用NCCL
    # init_method = 'env://' 表示通过环境变量获取MASTER_ADDR和MASTER_PORT
    dist.init_process_group(
        backend = backend,
        init_method='env://',
        wold_size=world_size,
        rank=rank
    )
    
    # 设置当前进程的GPU设备
    torch.cuda.set_device(rank)
    
    # 打印NCCL相关信息（用于调试）
    if rank == 0 :
        print(f"使用后端：{dist.get_backend()}")
        print(f"world_size大小：{dist.get_world_size()}")
        print(f"NCCL版本：{torch.cuda.nccl.version()}")
        
        
def cleanup():
    '''清理分布式环境'''
    dist.destroy_process_group()
    
# 4.训练函数
def train(rank,world_size,args):
    '''每个进程的训练逻辑'''
    # 1.设置分布式环境
    setup(rank,world_size)
    
    # 2.创建模型并移动到对应GPU
    model = SimpleModel().to(rank)
    
    # 3.使用DDP包装模型
    # DDP会自动使用NCCL进行梯度同步
    ddp_model = DDP(
        model,
        device_ids = [rank],
        output_device = rank,
        find_unused_parameters = False # 设置为True如果模型有未使用的参数
    )
    
    # 4.创建数据集和数据加载器
    dataset = RandomDataset(num_samples = args.num_samples)
    
    # 使用DistribuedSampler确保每个进程使用不同的数据
    sampler = DistributedSampler(
        dataset,
        num_replicas = world_size,
        rank = rank,
        shuffle = True,
        seed = 42
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        sampler=sampler,
        num_workers=4,
        pin_memory=True, # 将数据加载到锁页内存中
    )
    # 5.定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(ddp_model.parameters(),lr=args.lr)
    
    # 6.训练循环
    for epoch in range(args.epochs):
        # 设置sampler的epoch，确保每个epoch数据被打乱，避免所有 GPU 在每个 epoch 看到相同顺序的数据。
        # 如果没有调用sampler.set_epoch(epoch)：所有GPU在每个epoch看到的数据顺序完全相同
        sampler.set_epoch(epoch)
        
        # 训练一个epoch
        ddp_model.train()
        total_loss = 0.0
        num_batches = 0
        
        for batch_idx,(data,target) in enumerate(dataloader):
            # 将数据移动到GPU
            data,target = data.cuda(rank),target.cuda(rank)
            
            # 前向传播
            output = ddp_model(data)
            loss = criterion(output,target)
            
            # 反向传播
            optimizer.zero_grad()
            loss.backgward()
            
            # 优化
            optimizer.step()
            
            total_loss +=loss.item()
            num_batches +=1
            
            # 打印训练信息（只在rank 0上打印，避免输出混乱）
            if rank == 0 and batch_idx % args.log_interval == 0 :
                print(f"epoch {epoch} [{batch_idx} / {len(dataloader)}]"
                      f"loss:{loss_item(0:.4f)}")
             
        # 计算平均损失（所有进程同步）
        avg_loss = total_loss / num_batches
        avg_loss_tensor = torch.tensor(avg_loss).cuda(rank)
        
        # 使用NCCL的all_reduce同步所有进程的损失
        dist.all_reduce(avg_loss_tensor,op=dist.ReduceOp.SUM)
        avg_loss_tensor /=world_size
        
        if rank == 0:
            print(f'Epoch {epoch} 完成，平均损失: {avg_loss_tensor.item():.4f}')
            
        # 同步所有进程（等待所有进程完成当前epoch
        dist.barrier()
        
    # 7.清理
    if rank ==0 :
        print('训练完成')
        
    cleanup()
    
# 5. 主函数
def main():
    parser = argparse.ArumentParser()
    parser.add_argument('--world-size',type=int,default=1,help="总进程数(GPU数量)")
    parser.add_argument('--rank',type=int,default=0,help='当前进程的排名（用于手动启动）')
    parser.add_argument('--master-addr',type=trl,default='localhost',help='主节点地址')
    parser.add_argument('--master-port',type=str,default='29500',help='主节点端口')
    parser.add_argument('--epochs',type=int,default=5,help='训练轮数')
    parser.add_argument('--batch-size',type=int,default=64,help='每个GPU的批次大小')
    parser.add_argument('--lr',type=float,default=0.001,help='学习率')
    parser.add_argument('--num-samples',type=int,default=10000,htlp='数据集大小')
    parser.add_argument('--log-interval',type=int,default=10,help='日志打印间隔')
    args = parser.parse_args()
    
    # 设置环境变量
    os.environ['MASTER_ADDR'] = args.master_addr
    os.environ['MASTER_PORT'] = args.master_port
    
    # 获取可用的GPU数量
    world_size =args_world_size if args.world_size > 0 else torch.cuda.device_count()
    
    print(f"启动训练，使用{world_size}个GPU")
    
    # 使用spawn起送多个进程
    mp.spawn(
        train,
        args = (world_size,args),
        nproc= world_size,
        join=True
    ) 
if __name__ =='__main__':
    main()

## 启动脚本

### 单机多卡启动：run_single_node.sh


In [ ]:
#!/bin/bash

# 设置NCCL环境变量(可选)
export NCCL_DEBUG = INFO # 开启调试信息
export NCCL_SOCKET_IFNAME=eth0 # 指定网卡用于通信
# export NCCL_P2P_LEVEL=NVL #  开启NVLlink加速（如果有）
export NCCL_ID_DISABLE=1 # 禁用infiniband（如果没有）

# 运行训练
python nccl_distributed_train.py \
    --world-size 4 \ # 使用4个GPU
    --epoch 10 \
    --batch-size 128 \
    --lr 0.001


### 多机多卡启动：run_multi_node.sh


In [ ]:
#!/bin/bash

# 节点1（主节点） 上运行
# MASTER_ADDR 设置为主节点的IP
# MASTER_PORT 设置为主节点的端口

# 节点1（主节点）
export MASTER_ADDR = '192.168.1.100' # 主节点的IP
export MASTER_PORT = '29500' 
export NCCL_DEBUG=INFO
export NCCL_SOCKET_IFNAME=ib0 # 使用infiniband
export NCCL_IB_DISABLE=0 #启动用infiniband
'''

在传统的网络通信中：
1.GPU 将数据拷贝到 CPU 内存 (Host Memory)。
2.CPU 再将数据拷贝到 网卡 (NIC)。
3.网卡发送数据。

GPU Direct RDMA (GDR) 技术允许：
1.GPU 显存 <-> 网卡显存 直接传输数据（通过 PCIe P2P 或 NVLink）。
2.绕过 CPU 内存。

决定 NCCL 在使用 InfiniBand (IB) 或 RoCE 网卡进行节点间通信时，允许数据在 GPU 显存和网卡之间直接传输的“拓扑距离”上限
简单来说，它告诉 NCCL：“如果网卡和 GPU 之间的物理连接不够‘亲密’（比如跨过了太多 PCIe 交换机），就不要强行使用 GPU Direct RDMA，降级为普通模式，以免性能反而下降或不稳定。”
'''
export NCCL_NET_GDR_LEVEL=5 # 启用GPU direct RDMA

# 启动节点1的8个GPU
python nccl_distributed_train.py \
    --world-size 16 \ # 总GPU数
    --rank 0 \ # 节点1的起始rank
    --epochs 10 \ 
    --batch-size 64

# 节点2上运行（修改rank和world-size保持不变）
export MASTER_ADDR='192.168.1.100' # 主节点使用的IP
export MASTER_PORT='29500'
export NCCL_DEBUG=INFO
export NCCL_SOCKET_IFNAME=ib0

# 启动节点2上的8个GPU
python nccl_distributed_train.py \
    --world-size 16 \
    --rank 8 \ # 节点2的起始rank
    --epoch 10 \ 
    --batch-size 64


## 使用torchrun启动（推荐）
### 单机多卡

In [ ]:
#!/bin/bash

export NCCL_DEBUG=INFO
export NCCL_SOCKET_IFNAME=eth0

# torchrun 会自动处理环境变量和进程启动
torchrun \
    --nnodes=1 \
    --nproc_per_node=4 \ # 单节点的GPU个数
    --master_addr=localhost \
    --master_port=29500 \
    nccl_distributed_train.py \ 
    --epoch 10 \ 
    --batch_size 128 \
    --lr 0.001

### 多机多卡
#### 节点1

In [ ]:
#!/bin/bash

export NCCL_DEBUG=INFO
export NCCL_SOCKET_IFNAME=ib0

torchrun \
    --nnodes=2 \
    --nproc_per_node=8 \
    --node_rank=0 \ 
    --master_addr = '192.168.1.100'
    --master_port = 29500 \ 
    nccl_distributed_train.py \ 
    --epoch 10 \ 
    --batch_size 64

#### 节点2


In [ ]:
#!/bin/bash

export NCCL_DEBUG=INFO
export NCCL_SOCKET_IFNAME=ib0

torchrun \
    --nnodes=2 \
    --nproc_per_node=8 \ 
    --node_rank=1 \ 
    --master_addr='192.168.1.100' \
    --master_port=29500 \ 
    nccl_distributed_train.py \ 
    --epochs 10 \ 
    --batch_size 64

## 测试NCCL通信的简单示例

In [ ]:
# test_nccl.py
import torch
import torch.distributed as dist
import os

def test_nccl_communication():
    """测试 NCCL 的基本通信功能"""
    
    # 初始化进程组
    dist.init_process_group(backend='nccl', init_method='env://')
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    
    # 创建张量
    tensor = torch.ones(1).cuda(rank) * rank
    
    print(f"Before all_reduce: rank {rank} has tensor {tensor.item()}")
    
    # 1. 测试 all_reduce (求和)
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    print(f"After all_reduce (SUM): rank {rank} has tensor {tensor.item()}")
    
    # 2. 测试 broadcast
    if rank == 0:
        tensor = torch.ones(1).cuda(rank) * 100
    dist.broadcast(tensor, src=0)
    print(f"After broadcast: rank {rank} has tensor {tensor.item()}")
    
    # 3. 测试 barrier
    dist.barrier()
    if rank == 0:
        print("All processes passed barrier!")
    
    dist.destroy_process_group()

if __name__ == '__main__':
    # 通过环境变量设置
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '29500'
    
    # 使用 torchrun 启动时，不需要手动 spawn
    test_nccl_communication()

#### 运行测试

In [ ]:
torchrun --nproc_per_node=4 test_nccl.py

# 内存类型比较


| 内存类型 | 特点 | 是否可分页 | GPU 访问速度 |
| :--- | :--- | :--- | :--- |
可分页内存 (Pageable Memory) |	常规内存，操作系统可以将其交换到磁盘 |	✅	| 慢（需要CPU中转） |
锁页内存 (Pinned Memory) |	锁定在物理内存中，不会被交换出去 |	❌ |	快（可直接DMA） |

## 工作原理示意图



In [ ]:
普通模式（pin_memory=False）：
CPU内存（可分页） → [CPU中转缓冲区] → GPU内存
           ↑_______________|
    需要CPU复制操作，速度较慢

锁页模式（pin_memory=True）：
CPU内存（锁页） → GPU内存（直接DMA传输）
           ↓
    无需CPU中转，速度更快

## 为什么pin_memory能加速

1. DMA 传输：锁页内存允许 GPU 直接通过 DMA（Direct Memory Access）从 CPU 内存读取数据，无需 CPU 参与复制过程。

2. 异步传输：在训练过程中，数据传输可以与 GPU 计算重叠进行。

## 与non_blocking=True配合使用

In [ ]:
# 当 pin_memory=True 时，PyTorch 会自动使用异步传输
class Trainer:
    def __init__(self, model, dataloader):
        self.model = model.cuda()
        self.dataloader = dataloader
        
    def train_step(self):
        for data, target in self.dataloader:
            # 当 pin_memory=True 时，使用 non_blocking=True
            data = data.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)
            
            # 计算和下一个batch的数据传输可以重叠
            output = self.model(data)
            loss = output.sum()
            loss.backward()

### 内存占用考虑

In [ ]:
import psutil
import torch

def check_pinned_memory_usage():
    """监控锁页内存使用情况"""
    
    # 创建大数据集
    dataset = torch.randn(10000, 3, 224, 224)
    
    # 创建 DataLoader
    loader = DataLoader(
        dataset,
        batch_size=64,
        pin_memory=True,
        num_workers=4
    )
    
    # 监控内存
    process = psutil.Process()
    initial_memory = process.memory_info().rss / 1024 / 1024  # MB
    
    # 遍历几个batch
    for i, batch in enumerate(loader):
        if i > 10:
            break
        # batch 在锁页内存中
    
    final_memory = process.memory_info().rss / 1024 / 1024
    print(f"初始内存: {initial_memory:.1f} MB")
    print(f"最终内存: {final_memory:.1f} MB")
    print(f"锁页内存占用: {final_memory - initial_memory:.1f} MB")

### 性能影响分析


| 场景 | pin_memory=False | pin_memory=True | 建议 |
| :--- | :--- | :--- | :--- |
小模型 + 大数据 |	慢	 |快	 |✅ 启用 |
大模型 + 小数据 |	影响较小 |	影响较小 |	⚠️ 可选 |
CPU 训练 |	相同	 |相同 |	❌ 无需启用 |
内存紧张 |	可接受	 |注意 |	⚠️ 谨慎使用 |



## 高级用法和调试

### 自定义锁页内存分配器

In [ ]:
import torch.cuda

# 创建自定义的锁页内存张量
pinned_tensor = torch.empty(1000, 1000, pin_memory=True)

# 或者从现有数据创建
data = torch.randn(100, 100)
pinned_data = data.pin_memory()  # 复制到锁页内存

# 异步传输到 GPU
gpu_data = pinned_data.cuda(non_blocking=True)

### 调试 pin_memory 问题

In [ ]:
def debug_pin_memroy_issues():
    '''诊断pin_memory相关问题'''
    
    print('===调试锁页内存===')
    
    # 检查系统是否支持
    print(f"pytorch版本：{torch.__version__}")
    print(f"CUDA可用：{torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU数量：{torch.cuda.device_count()}")
        print(f"当前GPU：{torch.cuda.current_device()}")
        print(f"GPU名称：{torch.cuda.get_device_name()}")
        
    #测试锁页内存分配
    try:
        test_tensor = torch.empty(10,10,pin_memory=True)
        print('锁页内存分配成功')
        
        # 测试异步传输
        test_tensor = test_tensor.cuda(non_blocking=True)
        print('异步传输成功')
    except Exception as e:
        print(f"锁页内存问题：{e}")
        
    # 检查内存限制
    import resource
    mem_limit = resource.getrlimit(resource.RLIMIT_MEMLOCK)
    print(f"锁页内存限制：软限制={mem_limit[0]/1024/1024:.1f}MB,  硬件限额：{mem_limit[1]/1024/1024:.1f}MB")
    
    if mem_limit[0] < 64 * 1024 * 1024 : # 64MB
        print('锁页内存限制可能过低，请考虑增加：')
        print('ulimit - l unlimited')

### 处理内存不足
- 根据可用内存动态调整 pin_memory 使用

In [ ]:
class MemoryAwareDataLoader:
    '''根据可用内存动态调整pin_memory使用'''
    def __init__(self,dataset,batch_size,num_workers):
        self.dataset = dataset
        self.batch_size = batch_size
        self.num_workers = num_workers
    
    def get_loader(self):
        # 检查可用内存
        import psutil
        available_memory = psutil.virtual_memroy().available / 1024/1024/1024/ # GB
        
        # 估算锁页内存需求
        sample = self.dataset[0]
        sample_memory = sample[0].element_size() * sample[0].nelement() / 1024 / 1024 # MB
        batch_memory = sample_memory * self.batch_size * self.num_workers / 1024 # GB
        
        # 根据可用内存决定是否使用锁页内存
        use_pin_memory = batch_memory < available_memory * 0.3 # 使用30%以下，使用锁页内存
        
        print(f"可用内存：{available_memory:.1f} GB")
        print(f"预计锁页内存需求：{batch_memory:.1f} GB")
        print(f"使用pin_memory：{use_pin_memory}")
        
        return DataLoader(
            self.dataset,
            batch_size =self.batch_size,
            num_workers = self.num_workers,
            pin_memory = use_pin_memory,
            prefetch_factor = 2
        )

### 性能监控脚本

In [9]:
import time
from contextlib import contextmanager

@contextmanager
def timeit(name):
    start = time.time()
    yield # 这里是关键
    elapsed = time.time() - start
    print(f"{name}:{elapsed*1000:.2f}ms")

#### yield 的核心作用

- yield 之前：进入上下文时执行的代码（记录开始时间）

- yield 之后：退出上下文时执行的代码（计算并打印耗时）

##### 执行流程详解


In [4]:
with timeit('数据处理耗时'): # 1.进入上下文
    # 2.yield暂停，执行这里的代码
    data = load_data() # 被计时的代码块
    process(data)      # 被计时的代码块
# 3.退出上下文，yield之后的代码执行


NameError: name 'load_data' is not defined

In [ ]:
with timeit("操作") as t:
    │
    ▼
执行 timeit 函数
    │
    ▼
start = time.time()  ←─┐  yield 之前：开始计时
    │                   │
    yield ←─────────────┼─ 暂停，返回控制权给 with 块
    │                   │
    │                   ▼
    │           执行 with 块内的代码
    │            data = load_data()
    │            result = process(data)
    │                   │
    │                   │  with 块执行完毕
    │                   │
    └─────────────→ 返回到 yield 之后
    │
    ▼
elapsed = time.time() - start  ←─ yield 之后：结束计时
print(f"操作: {elapsed} ms")

#### 支持异常处理

In [5]:
@contextmanager
def timeit(name, verbose=True):
    """增强版的计时器，支持异常处理和返回值"""
    start = time.time()
    
    try:
        yield  # with 块内的代码在这里执行
    except Exception as e:
        # 如果 with 块内发生异常，记录失败信息
        elapsed = time.time() - start
        print(f"❌ [{name}] 失败! 耗时: {elapsed*1000:.2f} ms, 错误: {e}")
        raise  # 重新抛出异常
    else:
        # 没有异常时执行
        elapsed = time.time() - start
        if verbose:
            print(f"✅ [{name}] 成功! 耗时: {elapsed*1000:.2f} ms")
        return elapsed  # 注意：contextmanager 不能直接返回，需要用其他方式

# 使用示例
with timeit("可能失败的操作"):
    # 正常情况
    time.sleep(0.1)
    
# 异常情况
try:
    with timeit("会失败的操作"):
        time.sleep(0.1)
        raise ValueError("出错了！")
except ValueError:
    pass

✅ [可能失败的操作] 成功! 耗时: 101.04 ms
❌ [会失败的操作] 失败! 耗时: 100.71 ms, 错误: 出错了！


#### 使用装饰器实现（用于函数计时）

In [7]:
def timeit_decorator(func):
    def wrapper(*args,**kwargs):
        start = time.time()
        result = func(*args,**kwargs) # 这里相当于yield的位置，with中执行代码
        elapsed = time.time() - start
        print(f"{func.__name__}:{elapsed*1000:.2f}ms")
        return result
    return wrapper
    
@timeit_decorator
def slow_function():
    time.sleep(0.1)
    
slow_function()

slow_function:100.80ms


#### 嵌套计时


In [11]:
@contextmanager
def timeit(name):
    start = time.time()
    print(f"计时开始：[{name}]")
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"计时结束 [{name}] 耗时：{elapsed*1000:.2f}ms")
        
# 嵌套使用
with timeit('整个流程'):
    time.sleep(0.2)
    with timeit('子任务1'):
        time.sleep(0.1)
        
        
    with timeit('子任务2'):
        time.sleep(0.15)
    time.sleep(0.1)

计时开始：[整个流程]
计时开始：[子任务1]
计时结束 [子任务1] 耗时：107.12ms
计时开始：[子任务2]
计时结束 [子任务2] 耗时：152.21ms
计时结束 [整个流程] 耗时：567.31ms
